# Notebook 03 — Latency, Throughput, and Memory Budgeting

This notebook builds the mechanical intuition behind open-model serving. We will model request flow, separate prefill from decode, explore batch effects, and make KV-cache budgeting concrete.

## What you will learn

- how to decompose request latency into understandable stages
- why prefill and decode scale differently
- how batch size changes throughput and p95 latency at the same time
- how KV cache turns context length into a memory planning problem

## The rule of thumb

When a team says “the model is slow,” ask four follow-up questions: is the pain queueing, prefill, decode, or memory pressure? Each one points to a different systems fix.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
random.seed(3)

NOTEBOOK_ROOT = ARTIFACT_ROOT / "module_03_latency_throughput_memory"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

def bytes_to_gb(value):
    return value / (1024 ** 3)

def percentile(values, pct):
    if len(values) == 0:
        return 0.0
    return float(np.percentile(values, pct))

def summarize_series(values):
    return {
        "mean": round(float(np.mean(values)), 2),
        "p50": round(percentile(values, 50), 2),
        "p95": round(percentile(values, 95), 2),
        "max": round(float(np.max(values)), 2),
    }

print("Artifact root:", NOTEBOOK_ROOT.resolve())

## Step 1: Describe requests as budgets

The most useful request description is not “chat” or “RAG.” It is a budget line: prompt tokens, output tokens, expected concurrency, and tolerance for waiting.

In [ ]:
request_df = pd.DataFrame([
    {"scenario": "short_chat", "prompt_tokens": 512, "output_tokens": 96, "arrival_rate_rps": 1.2, "batch_size": 2},
    {"scenario": "rag_answer", "prompt_tokens": 2400, "output_tokens": 128, "arrival_rate_rps": 0.9, "batch_size": 4},
    {"scenario": "code_edit", "prompt_tokens": 1800, "output_tokens": 220, "arrival_rate_rps": 1.1, "batch_size": 3},
    {"scenario": "batch_extract", "prompt_tokens": 6400, "output_tokens": 80, "arrival_rate_rps": 0.5, "batch_size": 8},
])
request_df["total_tokens"] = request_df["prompt_tokens"] + request_df["output_tokens"]
request_df

## Step 2: Create a small model profile

We do not need a real runtime to learn the shape of the problem. A simplified model profile is enough to turn token counts into latency and memory estimates.

In [ ]:
model_profile = {
    "name": "generic-open-7b",
    "layers": 32,
    "kv_heads": 8,
    "head_dim": 128,
    "bytes_per_kv_element": 2,
    "prefill_coeff": 0.030,
    "decode_base_ms": 1.6,
    "decode_ctx_coeff": 0.0026,
}
model_profile

## Step 3: Estimate one request lifecycle

The synthetic estimator below is intentionally simple. Its job is not to be exact. Its job is to preserve the qualitative behavior that later runtimes optimize.

In [ ]:
def estimate_lifecycle(prompt_tokens, output_tokens, batch_size, prefix_hit_rate=0.0):
    effective_prompt_tokens = max(1.0, prompt_tokens * (1.0 - prefix_hit_rate))
    queue_ms = max(0.0, 11.0 * (batch_size - 1))
    tokenize_ms = 5.0 + 0.009 * prompt_tokens
    prefill_ms = 18.0 + model_profile["prefill_coeff"] * (effective_prompt_tokens ** 1.13)
    decode_per_token_ms = model_profile["decode_base_ms"] + model_profile["decode_ctx_coeff"] * prompt_tokens + 0.05 * max(0, batch_size - 1)
    decode_ms = output_tokens * decode_per_token_ms
    post_ms = 7.0 + 0.015 * output_tokens
    total_ms = queue_ms + tokenize_ms + prefill_ms + decode_ms + post_ms
    return {
        "queue_ms": round(queue_ms, 2),
        "tokenize_ms": round(tokenize_ms, 2),
        "prefill_ms": round(prefill_ms, 2),
        "decode_ms": round(decode_ms, 2),
        "decode_per_token_ms": round(decode_per_token_ms, 3),
        "post_ms": round(post_ms, 2),
        "total_ms": round(total_ms, 2),
    }

In [ ]:
lifecycle_rows = []
for row in request_df.to_dict("records"):
    lifecycle_rows.append({**row, **estimate_lifecycle(row["prompt_tokens"], row["output_tokens"], row["batch_size"], prefix_hit_rate=0.25)})

lifecycle_df = pd.DataFrame(lifecycle_rows)
lifecycle_df[["scenario", "queue_ms", "prefill_ms", "decode_ms", "total_ms"]]

## Step 4: Prefill gets harder as the prompt grows

Long prompts hurt even before the first generated token appears. This is why retrieval-heavy or agent-heavy workloads need explicit prefill thinking.

In [ ]:
prefill_rows = []
for prompt_tokens in [128, 512, 1024, 2048, 4096, 8192, 16384]:
    metrics = estimate_lifecycle(prompt_tokens, output_tokens=128, batch_size=4, prefix_hit_rate=0.0)
    prefill_rows.append({
        "prompt_tokens": prompt_tokens,
        "prefill_ms": metrics["prefill_ms"],
        "decode_ms": metrics["decode_ms"],
        "total_ms": metrics["total_ms"],
    })

prefill_df = pd.DataFrame(prefill_rows)
prefill_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prefill_df["prompt_tokens"], prefill_df["prefill_ms"], marker="o", label="prefill")
ax.plot(prefill_df["prompt_tokens"], prefill_df["decode_ms"], marker="s", label="decode")
ax.plot(prefill_df["prompt_tokens"], prefill_df["total_ms"], marker="^", label="total")
ax.set_xscale("log", base=2)
ax.set_xlabel("prompt tokens")
ax.set_ylabel("milliseconds")
ax.set_title("Prompt length shifts latency into prefill")
ax.legend()
plt.show()

## Step 5: Decode scales with output length and active context

Decode is token-by-token work. Longer answers create more decode steps, and longer contexts slow each step because more history must be carried forward.

In [ ]:
decode_rows = []
for output_tokens in [32, 64, 128, 256, 384]:
    for batch_size in [1, 2, 4, 8]:
        metrics = estimate_lifecycle(prompt_tokens=2048, output_tokens=output_tokens, batch_size=batch_size, prefix_hit_rate=0.0)
        decode_rows.append({
            "output_tokens": output_tokens,
            "batch_size": batch_size,
            "decode_ms": metrics["decode_ms"],
            "decode_per_token_ms": metrics["decode_per_token_ms"],
            "total_ms": metrics["total_ms"],
        })

decode_df = pd.DataFrame(decode_rows)
decode_df.head(12)

## Step 6: Batch size changes throughput and latency together

Bigger batches usually help tokens per second, but they can also increase queueing and time to first token. Good serving is therefore a balancing act, not a monotonic optimization.

In [ ]:
def estimate_batch_metrics(prompt_tokens, output_tokens, batch_size, arrival_rate_rps):
    metrics = estimate_lifecycle(prompt_tokens, output_tokens, batch_size=batch_size, prefix_hit_rate=0.1)
    total_s = metrics["total_ms"] / 1000.0
    throughput_tps = round((batch_size * output_tokens) / max(total_s, 1e-6), 2)
    capacity_rps = round(batch_size / max(total_s, 1e-6), 2)
    utilization = round(min(1.5, arrival_rate_rps / max(capacity_rps, 1e-6)), 2)
    return {
        "batch_size": batch_size,
        "latency_ms": metrics["total_ms"],
        "throughput_tps": throughput_tps,
        "capacity_rps": capacity_rps,
        "utilization": utilization,
    }

batch_metrics_df = pd.DataFrame([estimate_batch_metrics(2048, 160, batch_size, arrival_rate_rps=1.8) for batch_size in [1, 2, 4, 8, 16]])
batch_metrics_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(batch_metrics_df["batch_size"], batch_metrics_df["throughput_tps"], marker="o", color="#54a24b", label="throughput_tps")
ax1.set_xlabel("batch size")
ax1.set_ylabel("tokens per second", color="#54a24b")
ax1.tick_params(axis="y", labelcolor="#54a24b")
ax2 = ax1.twinx()
ax2.plot(batch_metrics_df["batch_size"], batch_metrics_df["latency_ms"], marker="s", color="#e45756", label="latency_ms")
ax2.set_ylabel("latency ms", color="#e45756")
ax2.tick_params(axis="y", labelcolor="#e45756")
plt.title("Bigger batches help throughput while raising latency")
plt.show()

## Step 7: Queueing is the missing part of many latency discussions

If arrival rate gets too close to service capacity, p95 latency can deteriorate dramatically even when the model itself has not changed.

In [ ]:
def simulate_arrivals(arrival_rate_rps, batch_size, prompt_tokens, output_tokens, n_requests=220, seed=0):
    rng = random.Random(seed)
    arrival_clock = 0.0
    server_free_ms = 0.0
    rows = []
    base_service_ms = estimate_lifecycle(prompt_tokens, output_tokens, batch_size=batch_size, prefix_hit_rate=0.15)["total_ms"]
    for request_id in range(n_requests):
        arrival_clock += rng.expovariate(arrival_rate_rps) * 1000.0
        start_ms = max(arrival_clock, server_free_ms)
        wait_ms = start_ms - arrival_clock
        service_ms = base_service_ms * rng.uniform(0.9, 1.12)
        finish_ms = start_ms + service_ms
        server_free_ms = finish_ms
        rows.append({
            "request_id": request_id,
            "wait_ms": round(wait_ms, 2),
            "latency_ms": round(wait_ms + service_ms, 2),
        })
    return pd.DataFrame(rows)

In [ ]:
queue_compare_rows = []
for arrival_rate in [0.8, 1.2, 1.6, 2.0]:
    simulated = simulate_arrivals(arrival_rate, batch_size=4, prompt_tokens=2048, output_tokens=160, n_requests=260, seed=int(arrival_rate * 100))
    queue_compare_rows.append({
        "arrival_rate_rps": arrival_rate,
        **summarize_series(simulated["wait_ms"]),
        "latency_p95_ms": round(percentile(simulated["latency_ms"], 95), 2),
    })

queue_compare_df = pd.DataFrame(queue_compare_rows)
queue_compare_df

## Step 8: KV cache turns context length into a memory planning problem

During generation, the runtime stores key and value tensors for prior tokens. That memory is not optional. It grows with sequence length, active sequences, and model shape.

In [ ]:
def kv_cache_gb(layers, kv_heads, head_dim, active_tokens, active_sequences, bytes_per_element=2):
    total_bytes = 2 * layers * kv_heads * head_dim * active_tokens * active_sequences * bytes_per_element
    return round(bytes_to_gb(total_bytes), 3)

model_catalog = pd.DataFrame([
    {"model": "open-3b", "layers": 28, "kv_heads": 8, "head_dim": 128, "weight_memory_gb_q4": 2.2},
    {"model": "open-7b", "layers": 32, "kv_heads": 8, "head_dim": 128, "weight_memory_gb_q4": 4.8},
    {"model": "open-14b", "layers": 40, "kv_heads": 8, "head_dim": 128, "weight_memory_gb_q4": 9.2},
])

kv_rows = []
for row in model_catalog.to_dict("records"):
    for active_tokens in [2048, 8192, 32768]:
        kv_rows.append({
            "model": row["model"],
            "active_tokens": active_tokens,
            "active_sequences": 8,
            "kv_cache_gb": kv_cache_gb(row["layers"], row["kv_heads"], row["head_dim"], active_tokens, active_sequences=8),
        })

kv_df = pd.DataFrame(kv_rows)
kv_df

In [ ]:
open7b_curve = []
for active_tokens in [1024, 2048, 4096, 8192, 16384, 32768]:
    open7b_curve.append({
        "active_tokens": active_tokens,
        "kv_cache_gb": kv_cache_gb(32, 8, 128, active_tokens, active_sequences=12),
    })

open7b_curve = pd.DataFrame(open7b_curve)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(open7b_curve["active_tokens"], open7b_curve["kv_cache_gb"], marker="o")
ax.set_xscale("log", base=2)
ax.set_xlabel("active tokens in cache")
ax.set_ylabel("KV cache GB")
ax.set_title("KV cache grows quickly with context")
plt.show()

## Step 9: Combine weight memory and KV cache into capacity planning

A machine budget must cover more than weights. It also needs KV cache, runtime overhead, and a safety margin. Otherwise the service works until the traffic spike arrives.

In [ ]:
hardware_tiers = pd.DataFrame([
    {"hardware": "16GB box", "usable_memory_gb": 13.5},
    {"hardware": "24GB box", "usable_memory_gb": 21.0},
    {"hardware": "48GB box", "usable_memory_gb": 42.0},
    {"hardware": "80GB box", "usable_memory_gb": 72.0},
])

capacity_rows = []
for hw in hardware_tiers.to_dict("records"):
    for model in model_catalog.to_dict("records"):
        kv_budget_gb = max(0.0, hw["usable_memory_gb"] - model["weight_memory_gb_q4"] - 2.0)
        tokens_per_sequence = int((kv_budget_gb * (1024 ** 3)) / max(1, 2 * model["layers"] * model["kv_heads"] * model["head_dim"] * 8 * 2))
        capacity_rows.append({
            "hardware": hw["hardware"],
            "model": model["model"],
            "kv_budget_gb": round(kv_budget_gb, 2),
            "approx_tokens_per_sequence_at_8_concurrent": int(tokens_per_sequence),
        })

capacity_df = pd.DataFrame(capacity_rows)
capacity_df

## Step 10: Prefix caching changes the effective prompt, not the output

A cache hit does not eliminate the request. It removes repeated prefill work. That means it is most valuable when many requests share a long, stable prefix.

In [ ]:
cache_rows = []
for prefix_hit_rate in [0.0, 0.25, 0.5, 0.75, 0.9]:
    metrics = estimate_lifecycle(prompt_tokens=4096, output_tokens=160, batch_size=6, prefix_hit_rate=prefix_hit_rate)
    cache_rows.append({
        "prefix_hit_rate": prefix_hit_rate,
        "prefill_ms": metrics["prefill_ms"],
        "decode_ms": metrics["decode_ms"],
        "total_ms": metrics["total_ms"],
    })

cache_effect_df = pd.DataFrame(cache_rows)
cache_effect_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cache_effect_df["prefix_hit_rate"], cache_effect_df["prefill_ms"], marker="o", label="prefill_ms")
ax.plot(cache_effect_df["prefix_hit_rate"], cache_effect_df["total_ms"], marker="s", label="total_ms")
ax.set_xlabel("prefix hit rate")
ax.set_ylabel("milliseconds")
ax.set_title("Cache hits reduce prefill-heavy requests")
ax.legend()
plt.show()

## Step 11: Memory pressure forces admission-control choices

If active sequences push KV cache beyond the budget, the runtime must shed load, shrink the batch, or shorten context. There is no free escape hatch.

In [ ]:
def admit_request(current_tokens, next_request_tokens, memory_budget_gb, model_row, active_sequences):
    projected_gb = kv_cache_gb(model_row["layers"], model_row["kv_heads"], model_row["head_dim"], current_tokens + next_request_tokens, active_sequences)
    return projected_gb <= memory_budget_gb

admission_rows = []
model_row = model_catalog[model_catalog["model"] == "open-7b"].iloc[0].to_dict()
current_tokens = 12000
for next_request_tokens in [512, 2048, 4096, 8192, 16384]:
    admission_rows.append({
        "next_request_tokens": next_request_tokens,
        "admit_on_6gb_kv_budget": admit_request(current_tokens, next_request_tokens, 6.0, model_row, active_sequences=10),
        "admit_on_12gb_kv_budget": admit_request(current_tokens, next_request_tokens, 12.0, model_row, active_sequences=10),
    })

admission_df = pd.DataFrame(admission_rows)
admission_df

## Step 12: Export a budgeting worksheet

The worksheet below saves the core tables so later notebooks can reuse them for benchmark and deployment exercises.

In [ ]:
lifecycle_df.to_csv(NOTEBOOK_ROOT / "lifecycle_summary.csv", index=False)
batch_metrics_df.to_csv(NOTEBOOK_ROOT / "batch_metrics.csv", index=False)
capacity_df.to_csv(NOTEBOOK_ROOT / "capacity_planning.csv", index=False)

budget_payload = {
    "model_profile": model_profile,
    "queue_compare": queue_compare_df.to_dict("records"),
    "cache_effect": cache_effect_df.to_dict("records"),
}
with (NOTEBOOK_ROOT / "budget_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(budget_payload, handle, indent=2)

print("Saved worksheet artifacts to", NOTEBOOK_ROOT.resolve())

## Exercises

1. Change the active sequence count in the KV-cache function and see how quickly memory budgets collapse.
2. Increase batch size until throughput gains stop compensating for latency growth.
3. Compare the cache hit plot with the prefill plot from Notebook 01 and explain the connection in your own words.

In [ ]:
budget_questions = pd.DataFrame([
    {"question": "Which scenario is most decode-heavy?", "your_answer": ""},
    {"question": "Which hardware tier first fits open-14b with useful KV headroom?", "your_answer": ""},
    {"question": "What is your preferred admission-control rule?", "your_answer": ""},
])
budget_questions

## Recap

Latency is not one number. It is a sum of queueing, prefill, decode, and post-processing. Throughput is not free. It competes with latency. Memory is not just weights. KV cache makes context a first-class capacity problem.

In [ ]:
assert prefill_df.iloc[-1]["prefill_ms"] > prefill_df.iloc[0]["prefill_ms"]
assert batch_metrics_df["throughput_tps"].max() >= batch_metrics_df["throughput_tps"].min()
assert kv_df["kv_cache_gb"].max() > kv_df["kv_cache_gb"].min()
print("Notebook 03 sanity checks passed.")